In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive

In [2]:
drive.mount('/content/drive')
IPUMS_csv_file_path = '/content/drive/MyDrive/INFO450/IPUMS.csv'
OCC_csv_file_path = '/content/drive/MyDrive/INFO450/OccupationCodes.csv'


Mounted at /content/drive


Imported all of the libraries and google drive. Mounted google drive so that we can all colaborate on the same notebook.

In [3]:
df = pd.read_csv('/content/drive/MyDrive/INFO450/IPUMS.csv')
OCC_df = pd.read_csv('/content/drive/MyDrive/INFO450/OccupationCodes.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/INFO450/OccupationCodes.csv'

converted csv to dataframes

In [ ]:
df.head()

In [ ]:
OCC_df.head()

In [ ]:
OCC_df['OCC Code'].info()

In [ ]:
df['OCC'].info()

Called both of the dataframes so we can see the all of the data and verify what needs to be cleaned. Verify what types of Dtype 'OCC' is in each of the df so we can convert them to match when we merge.

In [ ]:
df = df.dropna()

Use df.drop function to drop any invalid rows

In [ ]:
df['OCC'] = df['OCC'].astype(str)
df = pd.merge(df, OCC_df, left_on='OCC', right_on='OCC Code', how='left')

set occ to a string then merged the data

In [ ]:
df = df[(df['UHRSWORKT'] >= 30) & (df['UHRSWORKT'] <= 60)]
df = df[(df['INCWAGE'] >= 10000) & (df['INCWAGE'] <= 200000)]
df = df[df['WKSWORK1'] > 0]

Made three filters

1.   Filters employees who whos usual hours are at least 30 hours and no more than 60
2.   Filters employees whos income is at least 10,000 and nor more than 200000 dollars
3.   Filters employees to ensure that they all worked more than one week


In [ ]:
df = df.drop(columns=['Unnamed: 3', 'OCC Code'])
df = df.dropna(subset=['Occupation Title']) # Drop anyone whose job didn't merge

Removed unnamed:3 because it was completely empty, dropped occ code since we already had one from the original IPUMS.csv.

Removed "noise variables"

In [ ]:
median_income = df['INCWAGE'].median()
df['HighEarner'] = (df['INCWAGE'] > median_income).astype(int)
df["HourlyEfficiency"] = df["INCWAGE"] / (df["WKSWORK1"] * df["UHRSWORKT"])
df["EdWorkIntensity"] = df["EDUC"] * df["UHRSWORKT"]


Call the merged df to verify that the occupation description is visible

In [ ]:
avg_inc_by_occupation = df.groupby('Occupation Title')['INCWAGE'].mean().reset_index()
avg_wage_by_education = df.groupby('EDUC')['INCWAGE'].mean().reset_index()
avg_hours_by_occupation = df.groupby('Occupation Title')['UHRSWORKT'].mean().reset_index()


In [ ]:
df

In [ ]:
import plotly.express as px

In [ ]:
horizontal_bar_chart = px.bar(df['Major Category'].value_counts().reset_index(),
                    x='count',
                    y='Major Category',
                    orientation='h',
                    title='Workforce Distribution: Number of Employees per Sector',
                    labels={'count': 'Number of Employees', 'Major Category': 'Job Sector'},
                    color='Major Category')
horizontal_bar_chart.update_layout(showlegend=False)
horizontal_bar_chart.show()

Created a horizontal bar for vis 1. Imported plotly instead of matplot lib since it is interactive. Px.bar to pull the bar chart and used the .value_counts().reset_index() functions to calculate the exact number of employees within each major category. Labeled the axis accordingly with job sector and number of employees. horizontal_bar_chart.update_layout(showlegend=False) gets rid of the legend on the right, the legend was redundant.

In [ ]:
histogram = px.histogram(df,
                    x='UHRSWORKT',
                    title='Workplace Culture: Distribution of Weekly Hours Worked',
                    labels={'UHRSWORKT': 'Hours Worked Per Week', 'count': 'Number of Employees', 'y': 'Number of Employees'},
                    color_discrete_sequence=['#1f77b4'])
histogram.update_layout(yaxis_title="Number of Employees")
histogram.show()

Used px.historgram to show how many hours people actually work per week. Unlike the first vis we did, I let plotly automatically count and group the raw data. I used the labels parameter as a translator to turn the code names, like UHRSWORKT, so that the user can read it as Hours Worked Per Week. Had to update the y axis since plotly couldn't translate it from the dictionary.

# Part 3 - Predictive Modeling

In [ ]:
model_df = df
model_df['HighEarner'] = df['HighEarner']
model_df.head()

In [ ]:
model_df['HighEarner'].value_counts()

In [ ]:
from sklearn.model_selection import train_test_split

X = model_df[[
    "AGE",
    "SEX",
    "RACE",
    "MARST",
    "OCC",
    "IND",
    "CLASSWKR",
    "EDUC",
    "UHRSWORKT",
    "WKSWORK1",
    "EdWorkIntensity"
]]
y = model_df["HighEarner"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

In [ ]:
numeric_features = ["AGE", "UHRSWORKT", "WKSWORK1", "EdWorkIntensity"]
categorical_features = [
    "SEX",
    "RACE",
    "MARST",
    "OCC",
    "IND",
    "CLASSWKR",
    "EDUC"
]

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"),categorical_features)
    ],
    remainder='passthrough'
)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

model = Pipeline([
    ("prep", preprocess),
    ("clf",DecisionTreeClassifier(max_depth=10,min_samples_leaf=20))
])

In [ ]:
model.fit(X_train,y_train)

In [ ]:
from sklearn.tree import export_graphviz
import graphviz

tree = model.named_steps["clf"]
feature_names = model.named_steps["prep"].get_feature_names_out()

dot = export_graphviz(
    tree,
    out_file=None,
    feature_names=feature_names,
    class_names=["Low", "High"],
    filled=True,
    rounded=True,
    special_characters=True,
    max_depth=4
)

graphviz.Source(dot)

In [ ]:
y_pred=model.predict(X_test)
y_pred

In [ ]:
y_test

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, accuracy_score,recall_score

print(confusion_matrix(y_test,y_pred))
print("accuracy", accuracy_score(y_test,y_pred))
print("precision", precision_score(y_test,y_pred))
print("recall",recall_score(y_test,y_pred))

# Part 4 - Streamlit App

In [ ]:
!pip install streamlit
import streamlit as st
import pandas as pd
import plotly.express as px

st.title("Workforce Dashboard")

# Bar chart sample
horizontal_bar_chart = px.bar(df['Major Category'].value_counts().reset_index(),
                    x='count',
                    y='Major Category',
                    orientation='h',
                    title='Workforce Distribution: Number of Employees per Sector',
                    labels={'count': 'Number of Employees', 'Major Category': 'Job Sector'},
                    color='Major Category')
horizontal_bar_chart.update_layout(showlegend=False)

st.plotly_chart(horizontal_bar_chart)

# Histogram sample
histogram = px.histogram(df,
                    x='UHRSWORKT',
                    title='Workplace Culture: Distribution of Weekly Hours Worked',
                    labels={'UHRSWORKT': 'Hours Worked Per Week', 'count': 'Number of Employees', 'y': 'Number of Employees'},
                    color_discrete_sequence=['#1f77b4'])
histogram.update_layout(yaxis_title="Number of Employees")
histogram.show()

st.plotly_chart(histogram)
st.dataframe(df)